# [STARTER] Exercise - Create an Agent with external API call enabled

In this exercise, you'll build an agent that can interact with external APIs to gather real-time data
and provide responses based on that information. You'll combine concepts from state management and
memory while adding the ability to make external API calls safely and effectively.


## Challenge

Your task is to create an agent that can make External API Calls:

- Implement tools that interact with real APIs
- Handle API responses and errors gracefully
- Use environment variables for API keys
- Process and format API data for user consumption

## Setup
First, let's import the necessary libraries:

In [1]:
import os
import random
from typing import List
import requests
from dotenv import load_dotenv

from lib.agents import Agent
from lib.messages import BaseMessage
from lib.tooling import tool

In [2]:
load_dotenv()

True

## Define API tools

Feel free to use any open service available through APIs.

Here are a few examples. You can follow the instructions given.
- https://jsonplaceholder.typicode.com/guide/
- https://www.exchangerate-api.com/
- https://openweathermap.org/

Or you can find one you're interested in here.
- https://github.com/public-apis/public-apis

In [3]:
# TODO: Define as many tools that access external APIs as you want
# Example:
@tool
def get_random_pokemon() -> dict:
    """Get a random Pokemon from the original 151"""
    URL = "https://pokeapi.co/api/v2/pokemon?limit=151"
    response = requests.get(URL)
    response.raise_for_status()
    return random.choice(response.json()['results'])

In [4]:
@tool
def get_exchange_rate(from_currency: str = "USD") -> dict:
    """
    Get latest exchange rates from a base currency
    args:
        from_currency (str): Base currency code (default: USD)
    """
    API_KEY = os.getenv("EXCHANGERATE_API_KEY")
    BASE_URL = "https://v6.exchangerate-api.com/v6"
    
    url = f"{BASE_URL}/{API_KEY}/latest/{from_currency}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

In [5]:
@tool
def get_weather(city: str) -> dict:
    """
    Get current weather for a city using OpenWeather API
    args:
        city (str): Name of the city to get weather for
    """
    API_KEY = os.getenv("OPENWEATHER_API_KEY")
    BASE_URL = "https://api.openweathermap.org/data/2.5/weather"
    
    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }
    
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json()

In [6]:
# TODO: Add all the tools you have defined
tools = [get_random_pokemon, get_exchange_rate, get_weather]

In [7]:
# TODO: Add instructions to your agent

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are an assistant that can help with:\n"
        "- Finding information about Pokemon using the get_random_pokemon tool.\n"
        "- Getting the latest exchange rates using the get_exchange_rate tool.\n"
        "- Retrieving current weather information for any city using the get_weather tool.\n"
        "Use the available tools to help answer questions about these topics.\n"
        "Maintain context across conversations within the same session."
    ),
    tools=tools
)

In [8]:
def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

## Run your Agent

In [9]:
# TODO: Change the query and then run your agent
query = "Tell me about a random Pokemon."
session_id = "external_tools"

In [10]:
run1 = agent.invoke(
    query=query, 
    session_id=session_id,
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are an assistant that can help with:
- Finding information about Pokemon using the get_random_pokemon tool.
- Getting the latest exchange rates using the get_exchange_rate tool.
- Retrieving current weather information for any city using the get_weather tool.
Use the available tools to help answer questions about these topics.
Maintain context across conversations within the same session., tool_calls = None)
 -> (role = user, content = Tell me about a random Pokemon., tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_iw2rAUuhx7gD9F8FXnmIQA9l', function=Function(arguments='{}', name='get_random_pokemon'), type=

In [11]:
query2 = "What's the exchange rate from USD to EUR?"
run2 = agent.invoke(
    query=query2, 
    session_id=session_id,
)
print("\nMessages from run 2:")
messages = run2.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 2:
 -> (role = system, content = You are an assistant that can help with:
- Finding information about Pokemon using the get_random_pokemon tool.
- Getting the latest exchange rates using the get_exchange_rate tool.
- Retrieving current weather information for any city using the get_weather tool.
Use the available tools to help answer questions about these topics.
Maintain context across conversations within the same session., tool_calls = None)
 -> (role = user, content = Tell me about a random Pokemon., tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_iw2rAUuhx7gD9F8FXnmIQA9l', function=Function(arguments='{}', name='get_random_pokemon'), type=

In [12]:
query3 = "What's the current weather in New York?"
run3 = agent.invoke(
    query=query3, 
    session_id=session_id,
)
print("\nMessages from run 3:")
messages = run3.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 3:
 -> (role = system, content = You are an assistant that can help with:
- Finding information about Pokemon using the get_random_pokemon tool.
- Getting the latest exchange rates using the get_exchange_rate tool.
- Retrieving current weather information for any city using the get_weather tool.
Use the available tools to help answer questions about these topics.
Maintain context across conversations within the same session., tool_calls = None)
 -> (role = user, content = Tell me about a random Pokemon., tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_iw2rAUuhx7gD9F8FXnmIQA9l', function=Function(arguments='{}', name='get_random_pokemon'), type=

## Check session histories

In [13]:
runs = agent.get_session_runs(session_id)
for i, run_object in enumerate(runs, 1):
    print(f"\n# Run {i}", run_object.metadata)
    print("Messages:")
    print_messages(run_object.get_final_state()["messages"])


# Run 1 {'run_id': '33b8cb6e-c174-4ee2-b814-47ce85554870', 'start_timestamp': '2026-01-11 17:43:58.273016', 'end_timestamp': '2026-01-11 17:44:02.410654', 'snapshot_counts': 5}
Messages:
 -> (role = system, content = You are an assistant that can help with:
- Finding information about Pokemon using the get_random_pokemon tool.
- Getting the latest exchange rates using the get_exchange_rate tool.
- Retrieving current weather information for any city using the get_weather tool.
Use the available tools to help answer questions about these topics.
Maintain context across conversations within the same session., tool_calls = None)
 -> (role = user, content = Tell me about a random Pokemon., tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_iw2rAUuhx7gD9F8FXnmIQA9l', function=Function(arguments='{}', name='get_random_pokemon'), type='function')])
 -> (role = tool, content = "{'name': 'lickitung', 'url': 'https://pokeapi.co/

In [14]:
runs = agent.get_session_runs(session_id)
for run_object in runs:
    print(run_object)
    for snp in run_object.snapshots:
        print(f"-> {snp}")
    print("\n")

Run('33b8cb6e-c174-4ee2-b814-47ce85554870')
-> Snapshot('5ff13ed8-14d7-4b11-b390-6c5f8685ce30') @ [2026-01-11 17:43:58.273691]: __entry__.State({'user_query': 'Tell me about a random Pokemon.', 'instructions': 'You are an assistant that can help with:\n- Finding information about Pokemon using the get_random_pokemon tool.\n- Getting the latest exchange rates using the get_exchange_rate tool.\n- Retrieving current weather information for any city using the get_weather tool.\nUse the available tools to help answer questions about these topics.\nMaintain context across conversations within the same session.', 'messages': [], 'current_tool_calls': None, 'session_id': 'external_tools'})
-> Snapshot('f659bf10-3a96-496d-a095-79d21abc7607') @ [2026-01-11 17:43:58.273993]: message_prep.State({'user_query': 'Tell me about a random Pokemon.', 'instructions': 'You are an assistant that can help with:\n- Finding information about Pokemon using the get_random_pokemon tool.\n- Getting the latest exch